# 4QDR.AI Universal Problem Solver — MCP Client Usage

This cookbook demonstrates how to connect to the **MCP server** (`mcp_server.py`)
via stdio JSON-RPC, list available tools, and call them programmatically.

## MCP Protocol Overview

The MCP server communicates over **stdin/stdout** using **JSON-RPC 2.0** messages.
Each message is a JSON object on a single line:

```json
{"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {...}}
```

Key methods:
- `initialize` — handshake, get server info
- `tools/list` — discover available tools with schemas
- `tools/call` — invoke a tool with parameters
- `ping` — health check

## Prerequisites

- Python 3.10+ (standard library only: `asyncio`, `json`, `subprocess`)
- Run from the project root so `mcp_server.py` is importable
- A valid Gemini session (run once with `--login` first)

In [ ]:
import asyncio
import json
import subprocess
import sys
import os
from pathlib import Path

ROOT = Path.cwd()
print(f"Project root: {ROOT}")

## Step 1: Launch the MCP server as a subprocess

We start `mcp_server.py` and communicate via its stdin/stdout.

In [ ]:
class MCPClient:
    """Minimal MCP client over stdio JSON-RPC."""
    
    def __init__(self):
        self.proc: subprocess.Popen | None = None
        self._request_id = 0
    
    def start(self):
        """Launch the MCP server subprocess."""
        self.proc = subprocess.Popen(
            [sys.executable, str(ROOT / 'mcp_server.py')],
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            cwd=str(ROOT)
        )
        print(f"MCP server started (PID: {self.proc.pid})")
    
    def send_request(self, method: str, params: dict | None = None) -> dict:
        """Send a JSON-RPC request and return the response."""
        self._request_id += 1
        request = {
            'jsonrpc': '2.0',
            'id': self._request_id,
            'method': method,
            'params': params or {}
        }
        request_line = json.dumps(request)
        print(f">> {method} (id={self._request_id})")
        
        self.proc.stdin.write(request_line + '\n')
        self.proc.stdin.flush()
        
        response_line = self.proc.stdout.readline().strip()
        if not response_line:
            raise ConnectionError("MCP server closed connection")
        
        return json.loads(response_line)
    
    def close(self):
        """Terminate the MCP server."""
        if self.proc:
            self.proc.terminate()
            self.proc.wait(timeout=5)
            print(f"MCP server terminated (PID: {self.proc.pid})")


client = MCPClient()
client.start()

## Step 2: Initialize the connection

The `initialize` handshake exchanges protocol version and capabilities.

In [ ]:
init_response = client.send_request('initialize', {
    'protocolVersion': '2024-11-05',
    'capabilities': {},
    'clientInfo': {
        'name': 'mcp-cookbook-client',
        'version': '1.0.0'
    }
})

print(json.dumps(init_response, indent=2, ensure_ascii=False))

## Step 3: List available tools

The `tools/list` call returns the full tool definition including the **semantic description**
that an LLM agent uses to decide whether to call this tool.

In [ ]:
tools_response = client.send_request('tools/list')
tools = tools_response.get('result', {}).get('tools', [])

print(f"Available tools: {len(tools)}")
for t in tools:
    print(f"\n  Name: {t['name']}")
    print(f"  Description: {t['description'][:120]}...")
    props = t.get('inputSchema', {}).get('properties', {})
    print(f"  Parameters: {', '.join(props.keys())}")

## Step 4: Call the gemini_web_chat tool

We invoke the tool with a prompt. The server launches Playwright, navigates to Gemini,
sends the prompt, and returns the structured response.

In [ ]:
call_response = client.send_request('tools/call', {
    'name': 'gemini_web_chat',
    'arguments': {
        'prompt': 'What are the three laws of robotics?',
        'model': 'fast',
        'tool': 'general',
        'headless': True,
        'timeout': 60
    }
})

result = call_response.get('result', {})
print(f"Success: {result.get('success')}")
print(f"Model: {result.get('model')}")
print(f"Duration: {result.get('duration', 0):.1f}s")
print(f"\n--- Answer (first 500 chars) ---\n{result.get('response', '')[:500]}")

## Step 5: Health check with ping

Use `ping` to verify the server is still alive.

In [ ]:
ping_response = client.send_request('ping')
print(f"Ping response: {ping_response.get('result')}")

## Step 6: Cleanup — terminate the server

In [ ]:
client.close()

## Full async example (end-to-end)

A complete function that initializes, calls, and cleans up in one go.

In [ ]:
async def call_gemini_via_mcp(prompt: str, timeout: int = 120) -> dict:
    """Send a prompt to Gemini via the MCP server."""
    c = MCPClient()
    try:
        c.start()
        
        # Initialize
        c.send_request('initialize', {
            'protocolVersion': '2024-11-05',
            'capabilities': {},
            'clientInfo': {'name': 'mcp-client', 'version': '1.0.0'}
        })
        
        # Call tool
        resp = c.send_request('tools/call', {
            'name': 'gemini_web_chat',
            'arguments': {
                'prompt': prompt,
                'headless': True,
                'timeout': timeout
            }
        })
        
        return resp.get('result', {})
    finally:
        c.close()


# Run it
result = await call_gemini_via_mcp("What is the speed of light in km/s?", timeout=60)
print(f"Response: {result.get('response', '')[:300]}...")

## Integration with LLM agents

To use this MCP server from an LLM agent framework:

- **Claude Desktop / Cursor / Continue.dev**: Add to your MCP config:
  ```json
  {
    "mcpServers": {
      "gemini-web-chat": {
        "command": "python",
        "args": ["path/to/mcp_server.py"],
        "env": {}
      }
    }
  }
  ```

- **OpenAI Agents SDK / LangChain / AutoGen**: Use `MCPClient` class above as a base
  to integrate `gemini_web_chat` into any agent tool loop.